In [1]:
# Import libraries
from opensearchpy import OpenSearch
from minio import Minio
import httpx
import os

In [2]:
# OpenSearch
os_client = OpenSearch(
    hosts=["http://localhost:9200"],
    use_ssl=False,
    verify_certs=False,
    timeout=30,
)

# MinIO
minio_client = Minio(
    endpoint="localhost:9002",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False,
)

In [ ]:
# Define the chunks indexed name
index = "arxiv-papers-chunks"
count = os_client.count(index=index)
print(f"Total Chunks: {count['count']}")

Total Chunks: 538


In [ ]:
# Get the fields of the indexed name
mapping = os_client.indices.get_mapping(index=index)
print(f"\nFields: {list(mapping[index]['mappings']['properties'].keys())}")


Fields: ['arxiv_id', 'chunk_text', 'chunk_vector', 'document_id', 'paper_id', 'project_id', 'title']


In [5]:
results = os_client.search(
    index="arxiv-papers-chunks",
    body={
        "size": 5,
        "query": {"match_all": {}},
        "_source": ["title", "document_id", "project_id", "arxiv_id", "chunk_text"],
    },
)

for hit in results["hits"]["hits"]:
    src = hit["_source"]
    print(f"ID: {hit['_id']}")
    print(f"  title: {src['title']}")
    print(f"  document_id: {src['document_id']}")
    print(f"  chunk_text: {src['chunk_text'][:120]}...")
    print()

ID: 23854461-17f0-47c6-9393-33527f84a226_0
  title: Introduction to Transformers: an NLP Perspective
  document_id: 23854461-17f0-47c6-9393-33527f84a226
  chunk_text: Introduction to Transformers
Introduction to Transformers: an NLP Perspective
Tong Xiao
xiaotong@mail.neu.edu.cn
NLP Lab...

ID: 23854461-17f0-47c6-9393-33527f84a226_1
  title: Introduction to Transformers: an NLP Perspective
  document_id: 23854461-17f0-47c6-9393-33527f84a226
  chunk_text: concepts of Transformers and present key tech-
niques that form the recent advances of these models. This includes a des...

ID: 23854461-17f0-47c6-9393-33527f84a226_2
  title: Introduction to Transformers: an NLP Perspective
  document_id: 23854461-17f0-47c6-9393-33527f84a226
  chunk_text: those concepts that are helpful for gaining a good
understanding of Transformers and their variants. We also summarize t...

ID: 23854461-17f0-47c6-9393-33527f84a226_3
  title: Introduction to Transformers: an NLP Perspective
  document_id: 23854461

In [10]:
doc_id = "23854461-17f0-47c6-9393-33527f84a226"

results = os_client.search(
    index="arxiv-papers-chunks",
    body={
        "size": 3,
        "query": {"term": {"document_id": doc_id}},
        "_source": ["title", "chunk_text"],
    },
)

print(f"Chunks for document {doc_id}: {results['hits']['total']['value']}")
for hit in results["hits"]["hits"]:
    print(f"\n--- {hit['_id']} ---")
    print(hit["_source"]["chunk_text"][:200])

Chunks for document 23854461-17f0-47c6-9393-33527f84a226: 227

--- 23854461-17f0-47c6-9393-33527f84a226_0 ---
Introduction to Transformers
Introduction to Transformers: an NLP Perspective
Tong Xiao
xiaotong@mail.neu.edu.cn
NLP Lab., Northeastern University, Shenyang, China
NiuTrans Research, Shenyang, China
J

--- 23854461-17f0-47c6-9393-33527f84a226_1 ---
concepts of Transformers and present key tech-
niques that form the recent advances of these models. This includes a description of the
standard Transformer architecture, a series of model refinements

--- 23854461-17f0-47c6-9393-33527f84a226_2 ---
those concepts that are helpful for gaining a good
understanding of Transformers and their variants. We also summarize the key ideas that
impact this field, thereby yielding some insights into the str


In [11]:
# Embed a query
query = "transformer attention mechanism"
resp = httpx.post(
    "https://api.openai.com/v1/embeddings",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    json={"model": "text-embedding-3-small", "input": query, "dimensions": 1024},
)
query_vector = resp.json()["data"][0]["embedding"]

# KNN search
results = os_client.search(
    index="arxiv-papers-chunks",
    body={
        "size": 5,
        "query": {
            "knn": {
                "chunk_vector": {
                    "vector": query_vector,
                    "k": 5,
                }
            }
        },
        "_source": ["title", "document_id", "arxiv_id", "chunk_text"],
    },
)

for hit in results["hits"]["hits"]:
    src = hit["_source"]
    print(f"Score: {hit['_score']:.4f} | {src['title']}")
    print(f"  {src['chunk_text'][:150]}...")
    print()

Score: 0.8109 | Introduction to Transformers: an NLP Perspective
  attention in Transformers makes it easier to deal with global contexts
and dependencies among words. Third, Transformers are very flexible architectur...

Score: 0.7933 | Introduction to Transformers: an NLP Perspective
  nsformer, it is usually achieved by examining the attention map and/or
output of an attention layer. Then, we construct a probing predictor (or probin...

Score: 0.7858 | Introduction to Transformers: an NLP Perspective
  denoted by {Hhead
1
, ..., Hhead
τ
}. The attention
model is then run τ times, each time on a head. Finally, the outputs of these model runs
are conca...

Score: 0.7775 | Introduction to Transformers: an NLP Perspective
  those concepts that are helpful for gaining a good
understanding of Transformers and their variants. We also summarize the key ideas that
impact this ...

Score: 0.7757 | Introduction to Transformers: an NLP Perspective
  rt by the fact that complex outputs can be
fo

In [8]:
# List buckets
print("Buckets:")
for b in minio_client.list_buckets():
    print(f"  {b.name} (created: {b.creation_date})")

# List objects in the app bucket
bucket = "researchhub-documents"
print(f"\nObjects in '{bucket}':")
for obj in minio_client.list_objects(bucket, recursive=True):
    size_kb = (obj.size or 0) / 1024
    print(f"  {obj.object_name}  ({size_kb:.1f} KB)")

Buckets:
  researchhub-documents (created: 2026-03-18 10:55:53.363000+00:00)

Objects in 'researchhub-documents':
  11318532-3fd6-4909-9aac-bcf1d3690bee/703586c2-5755-4cab-9c1a-bb160317ed4f/68405.VOR.pdf  (4728.2 KB)


In [9]:
import tempfile, pymupdf

# Pick the first object from the bucket
objects = list(minio_client.list_objects("researchhub-documents", recursive=True))
if not objects:
    print("No files in bucket")
else:
    key = objects[0].object_name
    print(f"Downloading: {key}")

    response = minio_client.get_object("researchhub-documents", key)
    pdf_bytes = response.read()
    response.close()
    response.release_conn()

    print(f"Size: {len(pdf_bytes) / 1024:.1f} KB")

    # Extract first page text
    with tempfile.NamedTemporaryFile(suffix=".pdf") as tmp:
        tmp.write(pdf_bytes)
        tmp.flush()
        doc = pymupdf.open(tmp.name)
        print(f"Pages: {len(doc)}")
        print(f"\n--- Page 1 preview ---\n{doc[0].get_text()[:500]}")
        doc.close()

Downloading: 11318532-3fd6-4909-9aac-bcf1d3690bee/703586c2-5755-4cab-9c1a-bb160317ed4f/68405.VOR.pdf
Size: 4728.2 KB
Pages: 20

--- Page 1 preview ---
Farmers’ credit risk evaluation with an explainable hybrid 
ensemble approach: A closer look in microfinance
Nana Chai a,1, Mohammad Zoynul Abedin b,1, Lian Yang c,*, Baofeng Shi d,e,**
a School of Economics and Management, Tongji University, Shanghai 200092, Shanghai, China
b Department of Accounting and Finance, School of Management, Swansea University, Bay Campus, Fabian Way, Swansea SA1 8EN, UK
c College of Economics and Management, Shandong Agricultural University, Taian 271000, Shandong, C


In [12]:
results = os_client.search(                                                                                                                
    index="arxiv-papers-chunks",                                                                                                           
    body={                                                                                                                                 
        "size": 5,                                                                                                                       
        "query": {"term": {"arxiv_id": ""}},
        "_source": ["title", "document_id", "project_id", "chunk_text"],
    },
)

print(f"Total uploaded doc chunks: {results['hits']['total']['value']}\n")

for hit in results["hits"]["hits"]:
    src = hit["_source"]
    print(f"ID: {hit['_id']}")
    print(f"  title: {src['title']}")
    print(f"  document_id: {src['document_id']}")
    print(f"  chunk_text: {src['chunk_text'][:120]}...")
    print()

Total uploaded doc chunks: 220

ID: 703586c2-5755-4cab-9c1a-bb160317ed4f_0
  title: 68405.VOR.pdf
  document_id: 703586c2-5755-4cab-9c1a-bb160317ed4f
  chunk_text: Farmers’ credit risk evaluation with an explainable hybrid 
ensemble approach: A closer look in microfinance
Nana Chai a...

ID: 703586c2-5755-4cab-9c1a-bb160317ed4f_1
  title: 68405.VOR.pdf
  document_id: 703586c2-5755-4cab-9c1a-bb160317ed4f
  chunk_text: hina
d College of Economics and Management, Northwest A&F University, Yangling 712100, Shaanxi, China
e Research Center ...

ID: 703586c2-5755-4cab-9c1a-bb160317ed4f_2
  title: 68405.VOR.pdf
  document_id: 703586c2-5755-4cab-9c1a-bb160317ed4f
  chunk_text: microcredit by reshaping credit risk evaluation 
models, especially targeting the group of farmers. Therefore, the paper...

ID: 703586c2-5755-4cab-9c1a-bb160317ed4f_3
  title: 68405.VOR.pdf
  document_id: 703586c2-5755-4cab-9c1a-bb160317ed4f
  chunk_text: cing model bias, and simplifying complex problems by 
learning di